In [ ]:
import json
import time
from tqdm import tqdm
from openai import OpenAI

# OpenAI API 配置
client = OpenAI(
    api_key="sk-bCeqVHwJt0yXU5VIarZPidGnjGK510UKA19Dqh6EuASQrCEG",
    base_url="https://lonlie.plus7.plus/v1"
)

# 每条数据生成几条增强数据
AUG_NUM = 5

# 构建 prompt，生成增强数据
def build_prompt(en_question, zh_question, pql_query, num):
    return f"""你是一个中英文双语的数据增强助手。请你基于下面这条数据，生成{num}条语义等价但表达方式不同的新数据，要求如下：

1. 改写中英文问题，表达自然、真实、语义一致；
2. 修改平台名（如 platform_A）和表名（如 table_X），由你自由合理命名；
3. 修改 SQL 查询语句（PQL_query），保持语义与改写问题一致；
4. 不要抄袭原问题中的平台名和表名；
5. 请严格以 JSON 数组形式返回，每个元素形如：
{{
  "question": "英文改写",
  "Chinese_question": "中文改写",
  "PQL_query": "对应SQL"
}}

下面是原始数据：
英文问题: {en_question}
中文问题: {zh_question}
PQL_query: {pql_query}
"""

# 调用大模型生成增强数据
def augment_sample(en_question, zh_question, pql_query, num=5):
    prompt = build_prompt(en_question, zh_question, pql_query, num)
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "你是一个中英文双语数据增强助手"},
                {"role": "user", "content": prompt}
            ],
            temperature=1.0,
            max_tokens=1500
        )
        content = response.choices[0].message.content.strip()
        json_start = content.find("[")
        json_data = content[json_start:]
        samples = json.loads(json_data)
        for s in samples:
            s["source"] = "augmented"
        return samples
    except Exception as e:
        print(f"❌ 大模型调用失败: {e}")
        return []

# 加载原始数据
with open("pql_dataset.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

augmented_data = {}

# 遍历所有场景
for scene_name, samples in raw_data.items():
    print(f"\n📂 处理场景：{scene_name}（共 {len(samples)} 条数据）")
    new_samples = []
    for i, sample in enumerate(tqdm(samples)):
        en_q = sample["question"]
        zh_q = sample["Chinese_question"]
        pql = sample["PQL_query"]

        # 添加原始数据，并标记来源
        new_samples.append({
            "question": en_q,
            "Chinese_question": zh_q,
            "PQL_query": pql,
            "source": "original"
        })

        # 添加增强数据
        augmented = augment_sample(en_q, zh_q, pql, num=AUG_NUM)
        new_samples.extend(augmented)

        # 限速：避免API限流
        time.sleep(1.5)

    augmented_data[scene_name] = new_samples

# 保存增强数据集
with open("augmented_dataset.json", "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, ensure_ascii=False, indent=2)

print("\n✅ 数据增强完成！文件已保存为 'augmented_dataset.json'")
